<a href="https://colab.research.google.com/github/GGSimmons1992/5VfneekyeG9soDiV/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1b: Classical Random Forest Training

In [ ]:
!pip install category_encoders

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
import pickle
from imblearn.over_sampling import SMOTENC
from os.path import exists
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from sklearn.feature_selection import chi2
from scipy.stats import spearmanr
import json
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Src/')
import dataPrep

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def splitBetweenTrainAndDev(df):
  train,dev = train_test_split(df,test_size=0.2,random_state=51)
  return train,dev

In [ ]:
def trainAndTestModel(model, train, dev):
  model.fit(train.drop(columns=['isSubscribed']), train['isSubscribed'])
  train_predictions = model.predict(train.drop(columns=['isSubscribed']))
  dev_predictions = model.predict(dev.drop(columns=['isSubscribed']))
  trainScore = f1_score(train['isSubscribed'], train_predictions)
  testScore = f1_score(dev['isSubscribed'], dev_predictions)
  return trainScore, testScore

In [ ]:
def createModel(criterion, nEstimators, maxDepth, maxFeatures):
  return rf(criterion=criterion, n_estimators=nEstimators, max_depth=maxDepth, max_features=maxFeatures)

In [ ]:
def createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary):
  return pd.DataFrame(scoreVsHyperparameterDictionary).sort_values(by=['testScore','trainScore'],ascending=False)

In [ ]:
def appendToHyperparameterDictionary(scoreVsHyperparameterDictionary, criterion, nEstimators, maxDepth, maxFeatures, trainScore, testScore):
  scoreVsHyperparameterDictionary['criterion'].append(criterion)
  scoreVsHyperparameterDictionary['nEstimators'].append(nEstimators)
  scoreVsHyperparameterDictionary['maxDepth'].append(maxDepth)
  scoreVsHyperparameterDictionary['maxFeatures'].append(maxFeatures)
  scoreVsHyperparameterDictionary['trainScore'].append(trainScore)
  scoreVsHyperparameterDictionary['testScore'].append(testScore)
  return scoreVsHyperparameterDictionary

In [ ]:
def modelExistsInDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  return exists(fullName)

In [ ]:
def saveModelToDrive(data,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  pickle.dump(data, open(fullName, 'wb'))

In [ ]:
def retrieveModelFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  return pickle.load(open(fullName, 'rb'))

In [ ]:
def adjustRange(dfColumn):
  # Ensure min and max are called as methods on a pandas Series
  col_min = dfColumn.min()
  col_max = dfColumn.max()

  if col_min != col_max:
    return np.arange(col_min, col_max + 1, 1) # Include col_max in the range

  # Handle the case where all values are the same
  return np.arange(2, 2 * col_max, 1)




In [ ]:
def main():
  train = dataPrep.retrieveCSVFromDrive("SalesReinforcerTrain.csv")
  test = dataPrep.retrieveCSVFromDrive("SalesReinforcerTest.csv")
  nColumns = len(train.columns)
  bestTestScore = 0

  possibleCriterion = ["gini","entropy","log_loss"]
  possibleNEstimators = np.arange(5, 100, 1)
  possibleMaxDepth = np.arange(2,80,1)
  possibleMaxFeatures = np.arange(2,120,1)

  scoreVsHyperparameterDictionary = {
      "criterion":[],
      "nEstimators":[],
      "maxDepth":[],
      "maxFeatures":[],
      "trainScore":[],
      "testScore":[]
  }

  for iteration in range(150):
    print(f"iteration: {iteration+1} of 150") # Updated print statement
    criterion = np.random.choice(possibleCriterion)
    nEstimators = np.random.choice(possibleNEstimators)
    maxDepth = np.random.choice(possibleMaxDepth)
    maxFeatures = np.random.choice(possibleMaxFeatures)
    model = createModel(criterion, nEstimators, maxDepth, maxFeatures)
    trainScore, testScore = trainAndTestModel(model, train, test)
    if (trainScore > .8) and (testScore > .8) and (testScore > bestTestScore):
      bestTestScore = testScore
      saveModelToDrive(model,"baseRandomForest.pkl")

    scoreVHyperparameterDictionary = appendToHyperparameterDictionary(scoreVsHyperparameterDictionary,
                                                                      criterion, nEstimators, maxDepth,
                                                                      maxFeatures, trainScore, testScore)
    if (iteration == 149):
      df = createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary)
      display(df)
      dataPrep.saveCSVToDrive(df,"BestHyperparameters.csv")
      if (modelExistsInDrive("baseRandomForest.pkl")):
        print('best model found')
      else:
        print('best model not found')
      return

    if (iteration % 10) == 9:
      bestFiveHyperparameterGroups = createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary).head(5)
      display(bestFiveHyperparameterGroups)
      possibleNEstimators = adjustRange(bestFiveHyperparameterGroups['nEstimators'])
      possibleMaxDepth = adjustRange(bestFiveHyperparameterGroups['maxDepth'])
      possibleMaxFeatures = adjustRange(bestFiveHyperparameterGroups['maxFeatures'])
      if (modelExistsInDrive("baseRandomForest.pkl")):
        print('best model found')
        testSet = dataPrep.retrieveCSVFromDrive("SalesReinforcerTest.csv")
        bestModel = retrieveModelFromDrive("baseRandomForest.pkl")
        test_predictions = bestModel.predict(testSet.drop(columns=['isSubscribed']))
        testScore = f1_score(testSet['isSubscribed'], test_predictions)
        print(f"test score: {testScore}")
        return
      scoreVsHyperparameterDictionary = {
          "criterion":[],
          "nEstimators":[],
          "maxDepth":[],
          "maxFeatures":[],
          "trainScore":[],
          "testScore":[]
      }

In [ ]:
if __name__ == "__main__":
  main()

iteration: 1 of 150
iteration: 2 of 150
iteration: 3 of 150
iteration: 4 of 150
iteration: 5 of 150
iteration: 6 of 150
iteration: 7 of 150
iteration: 8 of 150
iteration: 9 of 150
iteration: 10 of 150


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,testScore
3,entropy,87,32,111,1.0,0.571429
4,entropy,94,65,118,1.0,0.571429
1,entropy,48,25,58,1.0,0.500000
2,log_loss,60,37,68,1.0,0.500000
5,gini,42,54,44,1.0,0.500000


iteration: 11 of 150
iteration: 12 of 150
iteration: 13 of 150
iteration: 14 of 150
iteration: 15 of 150
iteration: 16 of 150
iteration: 17 of 150
iteration: 18 of 150
iteration: 19 of 150
iteration: 20 of 150


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,testScore
1,gini,88,42,76,1.0,0.571429
5,entropy,60,27,117,1.0,0.571429
0,log_loss,81,27,81,1.0,0.500000
2,entropy,83,29,72,1.0,0.500000
3,gini,60,32,84,1.0,0.500000


iteration: 21 of 150
iteration: 22 of 150
iteration: 23 of 150
iteration: 24 of 150
iteration: 25 of 150
iteration: 26 of 150
iteration: 27 of 150
iteration: 28 of 150
iteration: 29 of 150
iteration: 30 of 150


...

,criterion,nEstimators,maxDepth,maxFeatures,trainScore,testScore
0,gini,77,37,109,1.0,0.571429
1,gini,74,38,110,1.0,0.571429
3,gini,76,35,107,1.0,0.571429
7,gini,74,35,109,1.0,0.571429
4,gini,75,37,107,1.0,0.500000


iteration: 71 of 150
iteration: 72 of 150
iteration: 73 of 150
iteration: 74 of 150
iteration: 75 of 150
iteration: 76 of 150
iteration: 77 of 150
iteration: 78 of 150
iteration: 79 of 150
iteration: 80 of 150


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,testScore
0,gini,74,36,107,1.0,0.571429
2,gini,74,36,107,1.0,0.571429
3,log_loss,75,37,107,1.0,0.571429
4,gini,77,38,107,1.0,0.571429
5,gini,76,37,108,1.0,0.571429


iteration: 81 of 150
iteration: 82 of 150
iteration: 83 of 150
iteration: 84 of 150


KeyboardInterrupt: 